# DX 704 Week 1 Project

This week's project will build a portfolio risk and return model, and make investing recommendations for hypothetical clients.
You will collect historical data, estimate returns and risks, construct efficient frontier portfolios, and sanity check the certainty of the maximum return portfolio.

The full project description and a template notebook are available on GitHub at the following link.

https://github.com/bu-cds-dx704/dx704-project-01


Feel free to use optimization tools or libraries (such as CVXOPT or scipy.optimize) to perform any calculations required for this mini project.

### Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Part 1: Collect Data

Collect historical monthly price data for the last 24 months covering 6 different stocks.
The data should cover 24 consecutive months including the last month that ended before this week's material was released on Blackboard.
To be clear, if a month ends between the Blackboard release and submitting your project, you do not need to add that month.

The six different stocks must include AAPL, SPY and TSLA.
At least one of the remaining 3 tickers must start with the same letter as your last name (e.g. professor Considine could use COIN).
This is to encourage diversity in what stocks you analyze; if you discuss this project with classmates, please make sure that you pick different tickers to differentiate your work.
Do not pick stocks with fewer than 24 consecutive months of price data.

In [33]:
import yfinance as yf
import pandas as pd
import numpy as np
import cvxopt
from cvxopt import matrix, solvers
import warnings

warnings.filterwarnings('ignore')
solvers.options['show_progress'] = False

tickers = ["AAPL", "SPY", "TSLA", "YELP", "MSFT", "GOOGL"]

# Fetch data and force the columns to match the exact order of the tickers list
df_daily = yf.download(tickers, start="2024-08-01", end="2026-09-01")['Close']
df_daily = df_daily[tickers] 

# Resample to the last valid trading day of each month
df_monthly = df_daily.resample('ME').last()
df_monthly.index = df_monthly.index.date
df_monthly.index.name = "date"

# Keep only the last 24 months
df_monthly = df_monthly.tail(24)



[*********************100%***********************]  6 of 6 completed


Save the data as a TSV file named "historical_prices.tsv" and include a header row with the column names "date" and the 6 stock ticker symbols.
The date should be the last trading day of the month, so it may not be the last day of the month.
For example, the last trading day of November 2024 was 2024-11-29.
The remaining columns should contain the adjusted closing prices of the corresponding stock tickers on that day.


In [34]:
# YOUR CHANGES HERE
df_monthly.to_csv("historical_prices.tsv", sep="\t")
df_monthly.head()

Ticker,AAPL,SPY,TSLA,YELP,MSFT,GOOGL
date,,,,,,
2024-09-30,231.067062,562.210449,261.630005,35.080002,423.608307,164.693512
2024-10-31,224.035873,557.193542,249.850006,34.139999,400.030701,169.916809
2024-11-30,235.620102,590.420898,345.160004,38.220001,417.709076,167.771881
2024-12-31,248.615768,576.215332,403.839996,38.700001,415.775726,188.195404
2025-01-31,234.299683,591.690308,404.600006,39.939999,409.423187,202.829529


Submit "historical_prices.tsv" in Gradescope.

## Part 2: Calculate Historical Asset Returns

Calculate the historical asset returns based on the price data that you previously collected.

In [19]:
df_returns = df_monthly.pct_change().dropna()



Save the data as a TSV file named "historical_returns.tsv" and include a header row with the column names "date" and the 6 stock ticker symbols.
Each row should have the date at the end of the month and the corresponding *relative* price changes.
For example, if the previous price was \$100 and the new price is \$110, the return value should be 0.10.
There should only be 23 rows of data in this file, since they are computed as the differences of 24 prices.

In [20]:
df_returns.to_csv("historical_returns.tsv", sep="\t")
df_returns.head()

Ticker,AAPL,GOOGL,MSFT,SPY,TSLA,YELP
date,,,,,,
2024-10-31,-0.030429,0.031715,-0.055659,-0.008924,-0.045025,-0.026796
2024-11-30,0.051707,-0.012623,0.044193,0.059633,0.381469,0.119508
2024-12-31,0.055155,0.121734,-0.004628,-0.024060,0.170008,0.012559
2025-01-31,-0.057583,0.077760,-0.015279,0.026856,0.001882,0.032041
2025-02-28,0.025872,-0.165376,-0.041618,-0.012695,-0.275877,-0.140961


Submit "historical_returns.tsv" in Gradescope.

## Part 3: Estimate Returns

Estimate the expected returns for each asset using the previously calculated return data.
Just compute the average (mean) return for each asset over your data set; do not use other estimators that have been mentioned.
This will serve as your estimate of expected return for each asset.

In [21]:
est_returns = df_returns.mean()
est_returns.name = "estimated_return"
est_returns.index.name = "asset"



Save the estimated returns in a TSV file named "estimated_returns.tsv" and include a header row with the column names "asset" and "estimated_return".

In [22]:
est_returns.to_csv("estimated_returns.tsv", sep="\t")
est_returns

asset
AAPL     0.015696
GOOGL    0.037147
MSFT     0.012032
SPY      0.014246
TSLA     0.027043
YELP    -0.014387
Name: estimated_return, dtype: float64

Submit "estimated_returns.tsv" in Gradescope.

## Part 4: Estimate Risk

Estimate the covariance matrix for the asset returns to understand how the assets move together.

In [23]:
est_cov = df_returns.cov()



Save the estimated covariances to a TSV file named "estimated_covariance.tsv".
The header row should have a blank column name followed by the names of the assets.
Each data row should start with the name of an asset for that row, and be followed by the individual covariances corresponding to that row and column's assets.
(This is the format of pandas's `to_csv` method with `sep="\t"` when used on a covariance matrix as computed in the examples.)

In [24]:
est_cov.to_csv("estimated_covariance.tsv", sep="\t")
est_cov

Ticker,AAPL,GOOGL,MSFT,SPY,TSLA,YELP
Ticker,,,,,,
AAPL,0.004014,0.002296,0.002296,0.001053,0.002829,-0.001928
GOOGL,0.002296,0.011690,0.002409,0.002395,0.005661,0.002499
MSFT,0.002296,0.002409,0.008978,0.001843,0.002710,0.001360
SPY,0.001053,0.002395,0.001843,0.001380,0.002818,0.000123
TSLA,0.002829,0.005661,0.002710,0.002818,0.026124,0.002113
YELP,-0.001928,0.002499,0.001360,0.000123,0.002113,0.010098


Submit "estimated_covariance.tsv" in Gradescope.

## Part 5: Construct the Maximum Return Portfolio

Compute the maximum return portfolio based on your previously estimated risks and returns.

In [35]:
max_return_asset = est_returns.idxmax()

# Use est_returns.index to guarantee the assets map properly to the data
allocations = [1.0 if asset == max_return_asset else 0.0 for asset in est_returns.index]

max_ret_port = pd.DataFrame({
    "asset": est_returns.index,
    "allocation": allocations
})


Save the maximum return portfolio in a TSV file named "maximum_return.tsv".
The header row should have two columns, "asset" and "allocation".
The allocation values should sum up to one.


In [36]:
max_ret_port.to_csv("maximum_return.tsv", sep="\t", index=False)
max_ret_port

,asset,allocation
0,AAPL,0.0
1,GOOGL,1.0
2,MSFT,0.0
3,SPY,0.0
4,TSLA,0.0
5,YELP,0.0


Submit "maximum_return.tsv" in Gradescope.

## Part 6: Construct the Minimum Risk Portfolio

Compute the minimum risk portfolio based on your previously estimated risks.

In [57]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

assets = est_returns.index
n_assets = len(assets)

# -----------------------------------------
# PART 6: Minimum Risk Portfolio
# -----------------------------------------

# The trick: Scale variance up by 100,000 to prevent the optimizer from stopping prematurely
def obj_variance_scaled(w, cov_matrix):
    return (w.T @ cov_matrix @ w) * 100000

bounds = tuple((0.0, 1.0) for _ in range(n_assets))
init_guess = np.repeat(1.0 / n_assets, n_assets)
cons_sum = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0}

res_min = minimize(
    obj_variance_scaled, 
    init_guess, 
    args=(est_cov.values,), 
    method='SLSQP', 
    bounds=bounds, 
    constraints=(cons_sum,),
    options={'ftol': 1e-15, 'maxiter': 5000}
)



Save the minimum risk portfolio in a TSV file named "minimum_risk.tsv".
The header row should have two columns, "asset" and "allocation".
The allocation values should sum up to one.


In [58]:
min_risk_weights = res_min.x
min_risk_port = pd.DataFrame({"asset": assets, "allocation": min_risk_weights})
min_risk_port.to_csv("minimum_risk.tsv", sep="\t", index=False)

# Store highly precise endpoints for Part 7
min_risk_return = np.sum(min_risk_weights * est_returns.values)
max_return = est_returns.max()

min_risk_port

,asset,allocation
0,AAPL,1.718505e-01
1,GOOGL,3.802876e-10
2,MSFT,1.742869e-10
3,SPY,6.898646e-01
4,TSLA,6.086658e-11
5,YELP,1.382849e-01


Submit "minimum_risk.tsv" in Gradescope.

## Part 7: Build Efficient Frontier Portfolios

Compute 101 portfolios along the mean-variance efficient frontier with evenly spaced estimated returns.
The first portfolio should be the minimum risk portfolio from part 4, and the last portfolio should be the maximum return portfolio from part 3.
The estimated return of each portfolio should be higher than the previous by one percent of the difference between the first and last portfolios.
That is, the estimated return of the portfolios should be similar to `np.linspace(min_risk_return, max_return, 101)`.


In [59]:
# -----------------------------------------
# PART 7: Efficient Frontier Portfolios
# -----------------------------------------

target_returns = np.linspace(min_risk_return, max_return, 101)
frontier_data = []

# Warm start: Begin the search from the exact minimum risk weights found in Part 6
current_guess = min_risk_weights.copy()

for i, target in enumerate(target_returns):
    cons_target = {'type': 'eq', 'fun': lambda w: np.sum(w * est_returns.values) - target}
    
    opt_res = minimize(
        obj_variance_scaled, 
        current_guess, 
        args=(est_cov.values,), 
        method='SLSQP', 
        bounds=bounds, 
        constraints=(cons_sum, cons_target),
        options={'ftol': 1e-15, 'maxiter': 5000}
    )
    
    weights = opt_res.x
    
    # At the final point (max return), equality constraints become infinitely narrow.
    # Force the known 100% maximum return allocation to prevent vertex errors.
    if i == 100:
        weights = np.zeros(n_assets)
        weights[np.argmax(est_returns.values)] = 1.0
        
    # Update guess for the next loop
    current_guess = weights
    
    # Calculate the actual UN-SCALED standard deviation for your output file
    actual_risk = np.sqrt(weights.T @ est_cov.values @ weights)
    
    row = {"index": i, "return": target, "risk": actual_risk}
    for j, ticker in enumerate(assets):
        row[ticker] = weights[j]
        
    frontier_data.append(row)



Save the portfolios in a TSV file named "efficient_frontier.tsv".
The header row should have columns "index", "return", "risk", and all the asset tickers.
Each data row should have the portfolio index (0-100), the estimated return of the portfolio, the estimated standard deviation (not variance) of the portfolio, and all the asset allocations (which should sum to one).

In [60]:
df_frontier = pd.DataFrame(frontier_data)
df_frontier.to_csv("efficient_frontier.tsv", sep="\t", index=False)
df_frontier.head()

,index,return,risk,AAPL,GOOGL,MSFT,SPY,TSLA,YELP
0,0,0.010536,0.033907,0.171850,3.802876e-10,1.742869e-10,0.689865,6.086658e-11,0.138285
1,1,0.010802,0.033921,0.168433,0.000000e+00,1.157882e-10,0.702749,1.596212e-10,0.128818
2,2,0.011068,0.033962,0.165016,0.000000e+00,1.061765e-11,0.715633,1.458086e-11,0.119351
3,3,0.011334,0.034031,0.161599,1.364707e-11,2.392742e-10,0.728517,2.902456e-14,0.109884
4,4,0.011600,0.034127,0.158181,0.000000e+00,2.669089e-11,0.741401,3.700650e-11,0.100417


Submit "efficient_frontier.tsv" in Gradescope.

## Part 8: Check Maximum Return Portfolio Stability

Check the stability of the maximum return portfolio by resampling the estimated risk/return model.

Repeat 1000 times -
1. Use `np.random.multivariate_normal` to generate 23 return samples using your previously estimated risks and returns.
2. Estimate the return of each asset using that resampled return history.
3. Check which asset had the highest return in those resampled estimates.

This procedure is a reduced and simplified version of the Michaud resampled efficient frontier procedure that takes uncertainty in the risk model into account.

In [31]:
# Part 8: Check Maximum Return Portfolio Stability
np.random.seed(42)
means = est_returns.values
cov_matrix = est_cov.values
iterations = 1000
sample_size = 23

highest_return_counts = {ticker: 0 for ticker in tickers}

for _ in range(iterations):
    # Generate 23 resampled monthly returns
    sim_returns = np.random.multivariate_normal(means, cov_matrix, sample_size)
    
    # Estimate the mean return for each asset
    sim_means = sim_returns.mean(axis=0)
    
    # Identify the highest estimated return
    best_idx = np.argmax(sim_means)
    best_asset = tickers[best_idx]
    highest_return_counts[best_asset] += 1

probabilities = {ticker: count / iterations for ticker, count in highest_return_counts.items()}

df_probs = pd.DataFrame(list(probabilities.items()), columns=["asset", "probability"])


Save a file "max_return_probabilities.tsv" with the distribution of highest return assets.
The header row should have columns "asset" and "probability".
There should be a data row for each asset and its sample probability of having the highest return based on those 1000 resampled estimates.


In [32]:
df_probs.to_csv("max_return_probabilities.tsv", sep="\t", index=False)
df_probs

,asset,probability
0,AAPL,0.065
1,SPY,0.489
2,TSLA,0.102
3,YELP,0.013
4,MSFT,0.326
5,GOOGL,0.005


Submit "max_return_probabilities.tsv" in Gradescope.

## Part 9: Acknowledgments

Make a file "acknowledgments.txt" documenting any outside sources or help on this project.
If you discussed this assignment with anyone, please acknowledge them here.
If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for.
If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy.
If no acknowledgments are appropriate, just write none in the file.


Submit "acknowledgments.txt" in Gradescope.

## Part 10: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

Submit "project.ipynb" in Gradescope.